In [26]:
# INSTALL & IMPORT
#!pip install geemap
import geemap

In [25]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-rakibul')

In [27]:
# LOAD BANGLADESH DISTRICTS (GADM)
districts = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))

In [28]:
# vDEFINE TIME RANGE
start_date = '2015-01-01'
end_date = '2023-12-31'

# SATELLITE DATA SOURCES

In [30]:
# 1. Rainfall (CHIRPS)
rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
    .filterDate(start_date, end_date) \
    .select("precipitation")

In [31]:
# 2. Vegetation (MODIS NDVI)
ndvi = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate(start_date, end_date) \
    .select("NDVI")

In [32]:
# 3. Land Surface Temperature
lst = ee.ImageCollection("MODIS/061/MOD11A2") \
    .filterDate(start_date, end_date) \
    .select("LST_Day_1km")

In [33]:
# 4. Soil Moisture
soil = ee.ImageCollection("NASA_USDA/HSL/SMAP10KM_soil_moisture") \
    .filterDate(start_date, end_date) \
    .select("ssm")

/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for NASA_USDA/HSL/SMAP10KM_soil_moisture! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_USDA_HSL_SMAP10KM_soil_moisture

  warnings.warn(warning, category=DeprecationWarning)


In [34]:
# FUNCTION TO COMPUTE MEAN PER DISTRICT
def zonal_stats(image_collection, band_name, region, scale=1000):
    """
    Compute mean satellite values per district.
    """

    image = image_collection.mean().select(band_name)

    result = image.reduceRegions(
        collection=region,
        reducer=ee.Reducer.mean(),
        scale=scale
    )

    return result

# EXTRACT FEATURES

In [35]:
# Rainfall
rain_stats = zonal_stats(rain, "precipitation", districts)

In [36]:
# NDVI
ndvi_stats = zonal_stats(ndvi, "NDVI", districts)

In [37]:
# LST
lst_stats = zonal_stats(lst, "LST_Day_1km", districts)

In [38]:
# Soil Moisture
soil_stats = zonal_stats(soil, "ssm", districts)

# MERGE ALL FEATURES

In [39]:
def merge_features(fc_list):
    merged = fc_list[0]
    for fc in fc_list[1:]:
        merged = merged.map(lambda f:
            f.set(fc.first().propertyNames(),
                  fc.filter(ee.Filter.eq('ADM2_NAME', f.get('ADM2_NAME'))).first())
        )
    return merged

# CONVERT TO PANDAS

In [40]:
rain_df = geemap.ee_to_df(rain_stats)
ndvi_df = geemap.ee_to_df(ndvi_stats)
lst_df = geemap.ee_to_df(lst_stats)
soil_df = geemap.ee_to_df(soil_stats)

In [42]:
# Clean Each DataFrame
rain_df = rain_df[['ADM2_NAME', 'mean']].rename(columns={'mean': 'rainfall'})
ndvi_df = ndvi_df[['ADM2_NAME', 'mean']].rename(columns={'mean': 'ndvi'})
lst_df  = lst_df[['ADM2_NAME', 'mean']].rename(columns={'mean': 'temperature'})
soil_df = soil_df[['ADM2_NAME', 'mean']].rename(columns={'mean': 'soil_moisture'})

In [43]:
# Merge Clean Data
df_satellite = rain_df.merge(ndvi_df, on="ADM2_NAME") \
                      .merge(lst_df, on="ADM2_NAME") \
                      .merge(soil_df, on="ADM2_NAME")

In [44]:
# CLEAN FINAL DATASET
df_satellite = df_satellite.rename(columns={
    "ADM2_NAME": "District",
    "precipitation": "rainfall",
    "NDVI": "ndvi",
    "LST_Day_1km": "lst",
    "ssm": "soil_moisture"
})

In [45]:
# SAVE DATASET
df_satellite.to_csv("gee_satellite_covariates_bd.csv", index=False)